In [ ]:
%pip install ipywidgets

In [ ]:
import io, warnings
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from scipy.ndimage import rotate as nd_rotate, zoom as nd_zoom, gaussian_filter
from scipy.signal import convolve2d
from ipywidgets import IntSlider, FloatSlider, Dropdown, VBox, HBox, Layout, Output
from IPython.display import display, Image, clear_output
warnings.filterwarnings('ignore')

_sl  = Layout(width="420px")
_sty = {"description_width": "150px"}

def fig2img(fig, dpi=100):
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0); plt.close(fig); return Image(data=buf.read())

def make_letter(text, size=64):
    fig, ax = plt.subplots(figsize=(2,2)); ax.axis('off')
    fig.text(0.23, 0.26, text, fontsize=100); fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    w, h = fig.canvas.get_width_height()
    data = buf.reshape(h,w,4)[:,:,:3].mean(2); plt.close(fig)
    data = (data-data.min())/(data.max()-data.min()+1e-10)
    data = (~data.astype(bool))[::-1].astype(float)
    return nd_zoom(data, size/data.shape[0])

def power_spectrum_2d(im):
    return np.log(np.abs(np.fft.fftshift(np.fft.fft2(im)))**2+1)

def sobel_x():   return np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float)
def sobel_y():   return np.array([[1,2,1],[0,0,0],[-1,-2,-1]], float)
def laplacian(): return np.array([[0,1,0],[1,-4,1],[0,1,0]], float)
def gaussian_kernel(size=9, sigma=2.0):
    ax = np.arange(-(size//2), size//2+1); k = np.exp(-ax**2/(2*sigma**2))
    k = np.outer(k,k); return k/k.sum()
def box_blur(size=5): return np.ones((size,size))/size**2
def sharpen():   return np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], float)

def make_ring(n=128, r=0.3, w=0.04):
    y,x = np.mgrid[0:n,0:n].astype(float)/n-0.5; R = np.sqrt(x**2+y**2)
    return ((R>r-w/2)&(R<r+w/2)).astype(float)
def make_hline(n=128, pos=0.0, width=0.03):
    y = np.linspace(-0.5,0.5,n)
    return (np.abs(y-pos)<width).astype(float)[:,None]*np.ones((n,n))
def make_dot(n=128, r=0.06):
    y,x = np.mgrid[0:n,0:n].astype(float)/n-0.5
    return (np.sqrt(x**2+y**2)<r).astype(float)

def relativistic_lambda(voltage_eV):
    return 12.264/np.sqrt(voltage_eV*(1+voltage_eV*0.98e-6))

def freq_mask(n, f1, f2=None, ftype='Low-pass'):
    y,x = np.mgrid[0:n,0:n].astype(float); y-=n//2; x-=n//2
    R = np.sqrt(x**2+y**2)/n
    if ftype=='Low-pass':  return (R<=f1).astype(float)
    elif ftype=='High-pass': return (R>=f1).astype(float)
    else: return ((R>=f1)&(R<=(f2 or f1))).astype(float)

def getbox(width=200, N=1000):
    sig=np.zeros(N); half=width//2; sig[N//2-half:N//2+half+1]=1; return sig

def fourier_series_reconstruct(signal, tot_freq):
    N=len(signal); x=np.arange(N); recon=np.zeros(N)
    for k in range(tot_freq):
        ak=2/N*np.sum(signal*np.cos(2*np.pi*k*x/N))
        bk=2/N*np.sum(signal*np.sin(2*np.pi*k*x/N))
        recon+=np.sqrt(ak**2+bk**2)*np.cos(2*np.pi*k*x/N-np.arctan2(bk,ak))
    recon-=recon.mean()-signal.mean(); return recon

Q64   = make_letter('Q', 64)
_Q128 = make_letter('Q', 128)
_Q128_ft = np.fft.fftshift(np.fft.fft2(_Q128))

# ── Widget factories ──────────────────────────────────────────────────────────
def wave_explorer():
    amp_sl  = FloatSlider(value=2.0,min=0.0,max=4.0,step=0.5,description="Amplitude A",style=_sty,layout=_sl)
    freq_sl = IntSlider(  value=4,  min=1,  max=12, step=1,  description="Frequency f", style=_sty,layout=_sl)
    pha_sl  = FloatSlider(value=0.0,min=-np.pi,max=np.pi,step=0.25,description="Phase φ (rad)",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        t=np.linspace(0,1,1000); f=amp_sl.value*np.cos(2*np.pi*freq_sl.value*t-pha_sl.value)
        fig,ax=plt.subplots(figsize=(10,3.5)); ax.plot(t,f,color='steelblue',lw=2)
        ax.axhline(0,color='k',lw=0.8,ls='--'); ax.set_xlim(0,1); ax.set_ylim(-4.5,4.5)
        ax.set_xlabel("t"); ax.set_ylabel("f(t)"); ax.grid(alpha=0.3)
        ax.set_title(f"f(t)={amp_sl.value:.1f}·cos(2π·{freq_sl.value}·t−{pha_sl.value:.2f})")
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [amp_sl,freq_sl,pha_sl]: w.observe(_u,names='value')
    display(VBox([amp_sl,freq_sl,pha_sl,out])); _u()

def fourier_series():
    w_sl=IntSlider(value=200,min=50,max=500,step=50,description="Box width (px)",style=_sty,layout=_sl)
    n_sl=IntSlider(value=15, min=1, max=80, step=1, description="# waves F",    style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        sig=getbox(w_sl.value); recon=fourier_series_reconstruct(sig,n_sl.value)
        xax=np.linspace(-500,500,len(sig))
        fig,axes=plt.subplots(1,2,figsize=(13,3.8))
        axes[0].plot(xax,sig,color='gray',lw=1.2,label='Box'); axes[0].plot(xax,recon,color='steelblue',lw=1.6,label=f'Recon F={n_sl.value}')
        axes[0].set_xlabel("x"); axes[0].set_ylabel("Amplitude"); axes[0].set_ylim(-0.4,1.5); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
        N=len(sig); ks=np.arange(1,n_sl.value+1); x=np.arange(N)
        amps=[np.sqrt((2/N*np.sum(sig*np.cos(2*np.pi*k*x/N)))**2+(2/N*np.sum(sig*np.sin(2*np.pi*k*x/N)))**2) for k in ks]
        axes[1].bar(ks,amps,color='steelblue',edgecolor='k',lw=0.4)
        axes[1].set_xlabel("Frequency k"); axes[1].set_ylabel("Amplitude"); axes[1].grid(alpha=0.3,axis='y')
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [w_sl,n_sl]: w.observe(_u,names='value')
    display(VBox([w_sl,n_sl,out])); _u()

def spectrum_explorer():
    a1=FloatSlider(value=1.0,min=0.0,max=5.0,step=0.5,description="Amp₁",style=_sty,layout=_sl)
    f1=IntSlider(  value=10, min=1,  max=100,step=1,  description="Freq₁",style=_sty,layout=_sl)
    a2=FloatSlider(value=1.0,min=0.0,max=5.0,step=0.5,description="Amp₂",style=_sty,layout=_sl)
    f2=IntSlider(  value=20, min=1,  max=100,step=1,  description="Freq₂",style=_sty,layout=_sl)
    a3=FloatSlider(value=1.0,min=0.0,max=5.0,step=0.5,description="Amp₃",style=_sty,layout=_sl)
    f3=IntSlider(  value=40, min=1,  max=100,step=1,  description="Freq₃",style=_sty,layout=_sl)
    dc=FloatSlider(value=5.0,min=-5.0,max=10.0,step=0.5,description="DC offset",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        N=1000; t=np.linspace(0,1,N)
        sig=dc.value+a1.value*np.cos(2*np.pi*f1.value*t)+a2.value*np.cos(2*np.pi*f2.value*t)+a3.value*np.cos(2*np.pi*f3.value*t)
        dft=np.fft.fft(sig); freqs=np.fft.fftfreq(N,1/N); amp=np.abs(dft)/N
        fig,axes=plt.subplots(1,2,figsize=(13,3.8))
        axes[0].plot(t,sig,color='steelblue',lw=1.4); axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Amplitude"); axes[0].grid(alpha=0.3)
        flim=min(100,max(f1.value,f2.value,f3.value)+20)
        axes[1].bar(freqs,amp,width=freqs[1]-freqs[0],color='steelblue',edgecolor='none')
        axes[1].set_xlabel("Frequency (Hz)"); axes[1].set_ylabel("Amplitude"); axes[1].set_xlim(-flim,flim); axes[1].grid(alpha=0.3)
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [a1,f1,a2,f2,a3,f3,dc]: w.observe(_u,names='value')
    display(VBox([HBox([a1,f1]),HBox([a2,f2]),HBox([a3,f3]),dc,out])); _u()

def wave_2d():
    fx=FloatSlider(value=3.0,min=0.0,max=10.0,step=0.5,description="fx (cycles/im)",style=_sty,layout=_sl)
    fy=FloatSlider(value=1.0,min=0.0,max=10.0,step=0.5,description="fy (cycles/im)",style=_sty,layout=_sl)
    ph=FloatSlider(value=0.0,min=-np.pi,max=np.pi,step=0.25,description="Phase φ",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        N=128; x=np.linspace(0,1,N); X,Y=np.meshgrid(x,x)
        wave=np.sin(2*np.pi*fx.value*X+2*np.pi*fy.value*Y+ph.value); ps=power_spectrum_2d(wave)
        fig,axes=plt.subplots(1,2,figsize=(10,4.5))
        axes[0].imshow(wave,cmap='RdBu',origin='lower',vmin=-1,vmax=1); axes[0].axis('off')
        axes[0].set_title(f"2D wave: fx={fx.value}, fy={fy.value}, φ={ph.value:.2f}")
        axes[1].imshow(ps,cmap='inferno',origin='lower'); axes[1].axis('off'); axes[1].set_title("Power spectrum")
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [fx,fy,ph]: w.observe(_u,names='value')
    display(VBox([fx,fy,ph,out])); _u()

def ft_patterns():
    n=128; examples=[(make_letter('Q',n),"Letter Q"),(make_ring(n),"Ring"),(make_hline(n),"Horiz. line"),
                     (make_dot(n),"Small dot"),(np.random.default_rng(0).standard_normal((n,n)),"Gaussian noise"),
                     (gaussian_filter(make_letter('Q',n),sigma=5),"Low-pass Q")]
    fig,axes=plt.subplots(2,len(examples),figsize=(3.5*len(examples),7))
    for col,(im,title) in enumerate(examples):
        ps=power_spectrum_2d(im); axes[0,col].imshow(im,cmap='gray',origin='lower'); axes[0,col].axis('off'); axes[0,col].set_title(title,fontsize=8)
        axes[1,col].imshow(ps,cmap='inferno',origin='lower'); axes[1,col].axis('off')
    axes[0,0].set_ylabel("Real space",fontsize=9); axes[1,0].set_ylabel("Fourier space",fontsize=9)
    plt.tight_layout(); display(fig2img(fig))

def ft_properties():
    dd=Dropdown(options=['Translation (+dx,+dy)','Rotation (30°)','Both'],value='Translation (+dx,+dy)',description="Property:",style=_sty)
    out=Output()
    def _u(_=None):
        c=dd.value
        if 'Translation' in c: Q2=np.roll(np.roll(_Q128,20,axis=0),30,axis=1)
        elif 'Rotation' in c:  Q2=nd_rotate(_Q128,30,reshape=False)
        else: Q2=nd_rotate(np.roll(np.roll(_Q128,20,axis=0),30,axis=1),30,reshape=False)
        F2=np.fft.fftshift(np.fft.fft2(Q2))
        fig,axes=plt.subplots(2,4,figsize=(16,8))
        for row,(im,ft,lab) in enumerate([(_Q128,_Q128_ft,'Original'),(Q2,F2,c)]):
            axes[row,0].imshow(im,cmap='gray',origin='lower'); axes[row,0].set_title(f"{lab}\nReal space"); axes[row,0].axis('off')
            axes[row,1].imshow(np.log(np.abs(ft)+1),cmap='inferno',origin='lower'); axes[row,1].set_title("FT amplitude"); axes[row,1].axis('off')
            axes[row,2].imshow(np.angle(ft),cmap='hsv',origin='lower',vmin=-np.pi,vmax=np.pi); axes[row,2].set_title("FT phase"); axes[row,2].axis('off')
        a1=np.log(np.abs(_Q128_ft)+1); a2=np.log(np.abs(F2)+1)
        axes[0,3].imshow(np.abs(a1-a2),cmap='hot',origin='lower'); axes[0,3].set_title("|Δ amplitude|"); axes[0,3].axis('off')
        axes[1,3].imshow(np.abs(np.angle(_Q128_ft)-np.angle(F2))%np.pi,cmap='hot',origin='lower'); axes[1,3].set_title("|Δ phase| mod π"); axes[1,3].axis('off')
        plt.suptitle(f"Fourier properties: {c}",fontsize=10); plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    dd.observe(_u,names='value'); display(VBox([dd,out])); _u()

def kernel_gallery():
    kernels={'Gaussian blur (σ=2)':gaussian_kernel(9,2.0),'Box blur (5×5)':box_blur(5),'Laplacian':laplacian(),'Sobel X':sobel_x(),'Sobel Y':sobel_y(),'Sharpen':sharpen()}
    fig,axes=plt.subplots(3,len(kernels),figsize=(3.5*len(kernels),9))
    for col,(name,kernel) in enumerate(kernels.items()):
        filtered=convolve2d(Q64,kernel,mode='same',boundary='wrap')
        axes[0,col].imshow(Q64,cmap='gray',origin='lower'); axes[0,col].axis('off'); axes[0,col].set_title(name,fontsize=7)
        axes[1,col].imshow(kernel,cmap='RdBu',origin='lower',vmin=-abs(kernel).max(),vmax=abs(kernel).max()); axes[1,col].axis('off'); axes[1,col].set_title(f"Kernel",fontsize=7)
        axes[2,col].imshow(filtered,cmap='gray',origin='lower'); axes[2,col].axis('off'); axes[2,col].set_title("Result",fontsize=7)
    axes[0,0].set_ylabel("Input",fontsize=9); axes[1,0].set_ylabel("Kernel",fontsize=9); axes[2,0].set_ylabel("Output",fontsize=9)
    plt.tight_layout(); display(fig2img(fig))

def kernel_explorer():
    dd =Dropdown(options=['Gaussian blur','Box blur','Laplacian','Sobel X','Sobel Y','Sharpen'],value='Gaussian blur',description="Kernel:",style=_sty)
    ksz=IntSlider(value=9,min=3,max=21,step=2,description="Kernel size",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        n=dd.value; k=ksz.value
        if n=='Gaussian blur': kernel=gaussian_kernel(k,k/6)
        elif n=='Box blur': kernel=box_blur(k)
        elif n=='Laplacian': kernel=laplacian()
        elif n=='Sobel X': kernel=sobel_x()
        elif n=='Sobel Y': kernel=sobel_y()
        else: kernel=sharpen()
        filtered=convolve2d(Q64,kernel,mode='same',boundary='wrap')
        ps_in=power_spectrum_2d(Q64)
        ps_k=power_spectrum_2d(np.pad(kernel,(32-kernel.shape[0]//2,32-kernel.shape[0]//2+1)))
        ps_out=power_spectrum_2d(filtered)
        fig,axes=plt.subplots(2,3,figsize=(13,8))
        axes[0,0].imshow(Q64,cmap='gray',origin='lower'); axes[0,0].axis('off'); axes[0,0].set_title("Input")
        axes[0,1].imshow(kernel,cmap='RdBu',origin='lower',vmin=-abs(kernel).max(),vmax=abs(kernel).max()); axes[0,1].axis('off'); axes[0,1].set_title(f"Kernel: {n}")
        axes[0,2].imshow(filtered,cmap='gray',origin='lower'); axes[0,2].axis('off'); axes[0,2].set_title("Filtered")
        axes[1,0].imshow(ps_in,cmap='inferno',origin='lower'); axes[1,0].axis('off'); axes[1,0].set_title("FT input")
        axes[1,1].imshow(ps_k,cmap='inferno',origin='lower'); axes[1,1].axis('off'); axes[1,1].set_title("FT kernel")
        axes[1,2].imshow(ps_out,cmap='inferno',origin='lower'); axes[1,2].axis('off'); axes[1,2].set_title("FT output")
        plt.suptitle("Convolution theorem",fontsize=10); plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [dd,ksz]: w.observe(_u,names='value')
    display(VBox([dd,ksz,out])); _u()

def frequency_filter():
    ftype=Dropdown(options=['Low-pass','High-pass','Band-pass'],value='Low-pass',description="Filter type:",style=_sty)
    fc1=FloatSlider(value=0.2,min=0.02,max=0.5,step=0.02,description="Cutoff f₁",style=_sty,layout=_sl)
    fc2=FloatSlider(value=0.4,min=0.02,max=0.5,step=0.02,description="Cutoff f₂ (band)",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        n=Q64.shape[0]; mask=freq_mask(n,fc1.value,fc2.value,ftype.value)
        F=np.fft.fftshift(np.fft.fft2(Q64)); Ff=F*mask
        filtered=np.real(np.fft.ifft2(np.fft.ifftshift(Ff)))
        fig,axes=plt.subplots(1,4,figsize=(16,4.2))
        axes[0].imshow(Q64,cmap='gray',origin='lower'); axes[0].axis('off'); axes[0].set_title("Original")
        axes[1].imshow(mask,cmap='Blues',origin='lower'); axes[1].axis('off'); axes[1].set_title(f"Mask ({ftype.value})")
        axes[2].imshow(np.log(np.abs(Ff)+1),cmap='inferno',origin='lower'); axes[2].axis('off'); axes[2].set_title("Filtered FT")
        axes[3].imshow(filtered,cmap='gray',origin='lower'); axes[3].axis('off'); axes[3].set_title("Result")
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [ftype,fc1,fc2]: w.observe(_u,names='value')
    display(VBox([ftype,fc1,fc2,out])); _u()

def ctf_explorer():
    df =FloatSlider(value=2.0,min=0.5,max=10.0,step=0.5,description="Defocus (µm)",style=_sty,layout=_sl)
    V  =FloatSlider(value=300,min=100,max=300, step=100, description="Voltage (kV)", style=_sty,layout=_sl)
    Cs =FloatSlider(value=2.7,min=0.0,max=5.0, step=0.1, description="Cs (mm)",      style=_sty,layout=_sl)
    phi=FloatSlider(value=0.0,min=-np.pi/2,max=np.pi/2,step=0.1,description="Phase shift Δφ",style=_sty,layout=_sl)
    Bf =FloatSlider(value=0.0,min=0.0,max=200.0,step=10.0,description="B-factor (Å²)",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        lam=relativistic_lambda(V.value*1e3); k=np.linspace(0,0.5,256)
        df_A=df.value*1e4; Cs_A=Cs.value*1e7
        gamma=(-np.pi/2)*Cs_A*lam**3*k**4+np.pi*lam*df_A*k**2
        ctf_curve=-np.sin(phi.value+gamma)
        if Bf.value>0: ctf_curve*=np.exp(-Bf.value*k**2)
        n=Q64.shape[0]; apix=1.0; ky=np.fft.fftfreq(n,apix); kx=np.fft.rfftfreq(n,apix)
        KX,KY=np.meshgrid(kx,ky); K2D=np.sqrt(KX**2+KY**2)*0.5/(n*apix)
        gamma2=(-np.pi/2)*Cs_A*lam**3*K2D**4+np.pi*lam*df_A*K2D**2
        ctf2d=-np.sin(phi.value+gamma2)
        if Bf.value>0: ctf2d*=np.exp(-Bf.value*K2D**2)
        Q_ctf=np.fft.irfftn(np.fft.rfftn(Q64)*ctf2d,Q64.shape)
        Q_noisy=Q_ctf+np.random.default_rng(1).standard_normal(Q64.shape)*0.3
        fig,axes=plt.subplots(1,4,figsize=(16,4.5))
        axes[0].plot(k,ctf_curve,color='steelblue',lw=1.8); axes[0].axhline(0,color='k',lw=0.8,ls='--')
        axes[0].set_xlabel("Spatial freq (Å⁻¹)"); axes[0].set_ylabel("CTF")
        axes[0].set_title(f"CTF (Δf={df.value} µm, V={V.value:.0f} kV)"); axes[0].set_ylim(-1.2,1.2); axes[0].grid(alpha=0.3)
        ps=np.log(np.abs(np.fft.fftshift(np.fft.fft2(Q_ctf)))**2+1)
        axes[1].imshow(Q64,cmap='gray',origin='lower'); axes[1].axis('off'); axes[1].set_title("True image")
        axes[2].imshow(Q_noisy,cmap='gray',origin='lower'); axes[2].axis('off'); axes[2].set_title("CTF-modulated")
        axes[3].imshow(ps,cmap='inferno',origin='lower'); axes[3].axis('off'); axes[3].set_title("Power spectrum (Thon rings)")
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [df,V,Cs,phi,Bf]: w.observe(_u,names='value')
    display(VBox([df,V,Cs,phi,Bf,out])); _u()

def ctf_correction():
    corr=Dropdown(options=['Phase flip','Full CTF (threshold)','Wiener filter'],value='Wiener filter',description="Correction:",style=_sty)
    snr =FloatSlider(value=1.0, min=0.01,max=10.0,step=0.1, description="SNR (Wiener)",style=_sty,layout=_sl)
    thr =FloatSlider(value=0.05,min=0.01,max=0.5, step=0.01,description="Threshold ε", style=_sty,layout=_sl)
    df2 =FloatSlider(value=2.0, min=0.5, max=10.0,step=0.5, description="Defocus (µm)",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        lam=relativistic_lambda(300e3); n=Q64.shape[0]; apix=1.0
        ky=np.fft.fftfreq(n,apix); kx=np.fft.rfftfreq(n,apix); KX,KY=np.meshgrid(kx,ky)
        K2D=np.sqrt(KX**2+KY**2)*0.5/(n*apix); df_A=df2.value*1e4; Cs_A=2.7e7
        ctf2d=-np.sin((-np.pi/2)*Cs_A*lam**3*K2D**4+np.pi*lam*df_A*K2D**2)
        F_obs=np.fft.rfftn(Q64)*ctf2d+np.random.default_rng(1).standard_normal(np.fft.rfftn(Q64).shape)*0.3
        m=corr.value
        if m=='Phase flip': Fc=F_obs*np.sign(ctf2d)
        elif m=='Full CTF (threshold)': Fc=np.where(np.abs(ctf2d)>=thr.value,F_obs/ctf2d,0.0)
        else: Fc=F_obs*ctf2d/(ctf2d**2+1.0/snr.value)
        Q_corr=np.fft.irfftn(Fc,Q64.shape); Q_ctf_i=np.fft.irfftn(F_obs,Q64.shape)
        fig,axes=plt.subplots(1,3,figsize=(12,4))
        axes[0].imshow(Q64,cmap='gray',origin='lower'); axes[0].axis('off'); axes[0].set_title("True image")
        axes[1].imshow(Q_ctf_i,cmap='gray',origin='lower'); axes[1].axis('off'); axes[1].set_title(f"CTF-modulated (Δf={df2.value} µm)")
        axes[2].imshow(Q_corr,cmap='gray',origin='lower'); axes[2].axis('off'); axes[2].set_title(f"Corrected: {m}")
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [corr,snr,thr,df2]: w.observe(_u,names='value')
    display(VBox([df2,corr,thr,snr,out])); _u()

print("Setup complete.")


# Practical: Fourier Optics and Image Processing

This practical covers the Fourier transform and its applications in image processing and cryo-EM optics.

> **To start the practicals:** click **Run All** (⏭) in the toolbar above.

---

## Part 1 – What is a wave?

A sinusoidal wave has three fundamental parameters:

$$
f(t) = A \cdot \cos(2\pi f \cdot t - \varphi)
$$

| Parameter | Symbol | Effect |
|-----------|--------|--------|
| Amplitude | $A$    | Height of the wave |
| Frequency | $f$    | Number of cycles per unit time |
| Phase     | $\varphi$ | Horizontal shift |

In [ ]:
wave_explorer()

---

## Part 2 – Fourier series

Any periodic signal can be expressed as a sum of sinusoidal waves (**Fourier series**):

$$
b(x) = A_0 + \sum_{k=1}^{F} A_k \cdot \cos\!\left(\frac{2\pi k x}{P} - \varphi_k\right)
$$

where $F$ is the maximum frequency, $P$ the period, and $A_k$, $\varphi_k$ the amplitude and phase at frequency $k$.

The amplitudes and phases are computed from the signal via the **Fourier coefficients**:

$$
a_k = \frac{2}{P}\sum_{i=0}^{P} b(i)\cos\!\frac{2\pi k i}{P}, \qquad
b_k = \frac{2}{P}\sum_{i=0}^{P} b(i)\sin\!\frac{2\pi k i}{P}, \qquad
A_k = \sqrt{a_k^2 + b_k^2}
$$

### The box function

A classic example is the box (rectangular) function:

$$
b(x) = \begin{cases} 1 & -a/2 < x < a/2 \\ 0 & \text{elsewhere} \end{cases}
$$

Its Fourier series contains only odd harmonics, and the coefficients decay as $1/k$. As we add more terms, the reconstruction improves but Gibbs ringing remains at the edges.

In [ ]:
fourier_series()

---

## Part 3 – Frequency spectrum (1D FFT)

Instead of computing Fourier coefficients term by term, the **discrete Fourier transform (DFT)** computes all of them at once:

$$
F(k) = \frac{1}{P} \sum_{m=0}^{P-1} b(m) \cdot e^{-i 2\pi k m / P}
$$

The fast Fourier transform (FFT) computes this in $O(N \log N)$ rather than $O(N^2)$. The **frequency spectrum** (plot of $|F(k)|$ vs $k$) reveals which frequencies are present in the signal.

### Nyquist sampling theorem

If a signal contains no frequencies higher than $W$ Hz, it is fully determined by samples taken every $1/(2W)$ seconds:

$$
f_\text{max} \leq \frac{1}{2 \Delta t} \quad \Leftrightarrow \quad \Delta t \leq \frac{1}{2 f_\text{max}}
$$

The **Nyquist frequency** $f_N = 1/(2\Delta t)$ is the highest frequency measurable at a given sampling rate. In EM images, the pixel size $\Delta x$ sets $f_N = 1/(2\Delta x)$, and only features larger than $2\Delta x$ can be resolved.

In [ ]:
spectrum_explorer()

```{admonition} Question 1
:class: seealso
Given a detector that records 15 frames per 100 s. What is the maximum frequency component that can be recovered? What does this imply for the pixel size required to resolve a feature of width 2 Å in a cryo-EM image?
```

---

## Part 4 – 2D Fourier analysis

Images are 2D signals. A 2D sinusoidal wave has the form:

$$
f(x,y) = A \cdot \sin(2\pi f_x x + 2\pi f_y y + \varphi)
$$

with separate spatial frequencies $f_x$ (cycles per pixel along $x$) and $f_y$ (cycles per pixel along $y$). The 2D Fourier transform of an image gives the amplitude and phase at every spatial frequency $(f_x, f_y)$.

### Properties

**Translation property**: Shifting an image in real space does not change the amplitude spectrum — only the phase spectrum.

**Rotation property**: Rotating an image rotates its Fourier transform by the same angle.

The interactive below shows 2D sine waves at different spatial frequencies, and the effects of translation and rotation on the Fourier transform of a real image.

In [ ]:
wave_2d()

### 2D FT of real images

Different image features produce characteristic Fourier signatures. A ring in real space produces a ring in Fourier space; a line produces a perpendicular line through the origin.

In [ ]:
ft_patterns()

### Translation and rotation properties

In [ ]:
ft_properties()

```{admonition} Question 2
:class: seealso
Play with the translation and rotation of the image and observe the amplitude spectrum. What changes and what stays the same in each case? Can you explain why?
```

---

## Part 5 – The convolution theorem

**Convolution** describes how one function modifies another by sweeping over all positions:

$$
(f * g)(y) = \int_{-\infty}^{\infty} f(x)\,g(y-x)\,dx
$$

In image processing, convolving an image with a **kernel** (a small matrix) applies a local operation: smoothing, sharpening, or edge detection. The **convolution theorem** states:

$$
\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}
$$

Convolution in real space equals *multiplication* in Fourier space. This makes filtering in Fourier space much faster for large kernels than direct spatial convolution.

### Common kernels

In [ ]:
kernel_gallery()

### Interactive kernel

In [ ]:
kernel_explorer()

```{admonition} Question 3
:class: seealso
What does the FT of a Gaussian blur kernel look like? What does this tell you about which spatial frequencies are attenuated? How does this compare to the Laplacian kernel?
```

---

## Part 6 – Low-pass and high-pass filtering in Fourier space

We can design filters by directly modifying the Fourier transform: set Fourier coefficients outside (or inside) a radius to zero before inverse-transforming. This is equivalent to convolving with the Fourier transform of the mask function.

In [ ]:
frequency_filter()

---

## Part 7 – The contrast transfer function (CTF)

In phase-contrast electron microscopy, the image intensity is related to the projected potential of the specimen through the **contrast transfer function (CTF)**. The CTF modulates the Fourier amplitudes as a function of spatial frequency $k$:

$$
\text{CTF}(k) = -\sin\!\left[\Delta\varphi + \frac{-\pi}{2}C_s\lambda^3 k^4 + \pi\lambda\Delta_f k^2\right]
$$

where:

| Symbol | Quantity |
|--------|----------|
| $k$ | spatial frequency (Å⁻¹) |
| $C_s$ | spherical aberration coefficient |
| $\lambda$ | relativistic electron wavelength |
| $\Delta_f$ | defocus (positive = underfocus) |
| $\Delta\varphi$ | additional phase shift (e.g. from phase plate) |

The electron wavelength depends on the accelerating voltage $V$ (in eV):

$$
\lambda = \frac{12.264}{\sqrt{V(1 + V \cdot 0.98 \times 10^{-6})}} \; \text{Å}
$$

Defocusing causes **phase reversals** at specific spatial frequencies (the CTF zeros). Frequencies near these zeros are not faithfully transferred to the image.

In [ ]:
ctf_explorer()

### CTF correction

To recover the true image from a CTF-modulated observation, we need to correct for the CTF. Three standard methods:

**Method 1 – Phase flipping**: Multiply Fourier amplitudes by the sign of the CTF. Brings all amplitudes to positive, but does not correct their magnitudes.

**Method 2 – Full CTF correction with threshold**: Divide by the CTF, but ignore frequencies where $|\text{CTF}| < \epsilon$ (near zeros) to avoid division instability.

**Method 3 – Wiener filter**: Divide by CTF with regularisation:
$$
\hat{F}(k) = \frac{F_\text{obs}(k) \cdot \text{CTF}(k)}{\text{CTF}(k)^2 + \text{SNR}^{-1}}
$$

In [ ]:
ctf_correction()

```{admonition} Question 4
:class: seealso
Compare the three CTF correction methods. What are the trade-offs?
- Phase flipping: what information is still lost?
- Full CTF correction with threshold: what happens to frequencies near the CTF zeros?
- Wiener filter: what is the optimal SNR for your simulated data?
```

---

## Summary

| Topic | Key concept |
|-------|-------------|
| Sinusoidal waves | $f(t) = A\cos(2\pi f t - \varphi)$; three parameters: $A$, $f$, $\varphi$ |
| Fourier series | Any periodic signal = sum of sinusoids; coefficients via $a_k$, $b_k$ integrals |
| FFT | Fast algorithm ($O(N\log N)$) for discrete FT; frequency spectrum = $|F(k)|$ |
| Nyquist | Sampling at $\Delta t$ can recover frequencies up to $1/(2\Delta t)$ |
| 2D FT | Generalises 1D; amplitude spectrum = $|F(f_x,f_y)|$; rotation and translation properties |
| Convolution theorem | $\mathcal{F}\{f*g\} = \mathcal{F}\{f\}\cdot\mathcal{F}\{g\}$; filter in Fourier space |
| CTF | $-\sin(\gamma(k))$; phase reversals at zeros; correct by phase flip, division, or Wiener filter |